# MBG Sentiment v2 — 5-fold regularized IndoBERTweet

This Kaggle notebook trains a stronger, more conservative v2 classifier for the Makan Bergizi Gratis public-opinion task.

Changes from v1:

- five-fold stratified cross-validation,
- averaged test probabilities instead of one split's predictions,
- classifier dropout `0.30`, encoder dropout `0.20`, attention dropout `0.15`,
- stronger weight decay `0.05`, label smoothing `0.05`, gradient clipping,
- early stopping and best-checkpoint restoration using validation Macro F1.

The competition `test.csv` is unlabeled. Metrics are calculated from out-of-fold predictions on the labeled training data; test predictions are exported for submission and the Streamlit artifact dashboard.

In [ ]:
import gc
import json
import random
import re
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

try:
    import emoji
except ImportError:
    emoji = None

warnings.filterwarnings("ignore")

SEED = 42
MODEL_NAME = "indolem/indobertweet-base-uncased"
LABELS = ["Negative", "Neutral", "Positive"]
N_SPLITS = 5
MAX_LENGTH = 160
LEARNING_RATE = 1.5e-5
NUM_EPOCHS = 6
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
LABEL_SMOOTHING = 0.05
MAX_GRAD_NORM = 1.0
CLASSIFIER_DROPOUT = 0.30
HIDDEN_DROPOUT = 0.20
ATTENTION_DROPOUT = 0.15
EXPORT_DIR = Path("/kaggle/working/indobertweet_mbg_v2")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL_NAME)
print("Regularization:", {
    "classifier_dropout": CLASSIFIER_DROPOUT,
    "hidden_dropout": HIDDEN_DROPOUT,
    "attention_dropout": ATTENTION_DROPOUT,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
})

## 1. Load the Kaggle data

Attach the competition dataset to the Kaggle notebook. The loader searches `/kaggle/input` for `train.csv` and `test.csv` so the competition folder name does not matter.

In [ ]:
def find_file(filename: str, root: str = "/kaggle/input") -> Path:
    matches = list(Path(root).rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r} under {root}. "
            "Attach the MBG competition dataset to this notebook."
        )
    return matches[0]


TRAIN_PATH = find_file("train.csv")
TEST_PATH = find_file("test.csv")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", TRAIN_PATH, train_df.shape, train_df.columns.tolist())
print("Test :", TEST_PATH, test_df.shape, test_df.columns.tolist())

LABEL_COL = "label" if "label" in train_df.columns else train_df.columns[-1]
ID_COL = "id" if "id" in test_df.columns else test_df.columns[0]
preferred_text = ["text", "comment", "komentar", "sentence", "content", "tweet"]
TEXT_COL = next(
    (column for column in preferred_text if column in train_df.columns),
    None,
)
if TEXT_COL is None:
    candidates = [
        column for column in train_df.columns
        if column not in {LABEL_COL, ID_COL}
        and train_df[column].dtype == "object"
    ]
    TEXT_COL = max(
        candidates,
        key=lambda column: train_df[column].fillna("").astype(str).str.len().mean(),
    )

train_df[TEXT_COL] = train_df[TEXT_COL].fillna("").astype(str)
test_df[TEXT_COL] = test_df[TEXT_COL].fillna("").astype(str)
train_df = train_df.dropna(subset=[LABEL_COL]).reset_index(drop=True)

print("ID column:", ID_COL)
print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)
print("Labels:", train_df[LABEL_COL].value_counts().to_dict())

## 2. Light social-text preprocessing and deterministic labels

The cleaning keeps sentiment-bearing words and emojis. It only normalizes URLs, mentions, whitespace, and emoji aliases.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
USER_RE = re.compile(r"(?<!\w)@\w+")
SPACE_RE = re.compile(r"\s+")


def clean_social_text(value: str) -> str:
    value = str(value).strip().lower()
    value = URL_RE.sub("HTTPURL", value)
    value = USER_RE.sub("@USER", value)
    if emoji is not None:
        value = emoji.demojize(value, delimiters=(" ", " "))
    return SPACE_RE.sub(" ", value).strip()


train_df["clean_text"] = train_df[TEXT_COL].map(clean_social_text)
test_df["clean_text"] = test_df[TEXT_COL].map(clean_social_text)

existing_labels = set(train_df[LABEL_COL].astype(str).unique())
ordered_labels = [label for label in LABELS if label in existing_labels]
if len(ordered_labels) != len(existing_labels):
    ordered_labels = sorted(existing_labels)

label2id = {label: index for index, label in enumerate(ordered_labels)}
id2label = {index: label for label, index in label2id.items()}
train_df["target"] = train_df[LABEL_COL].astype(str).map(label2id)

assert train_df["target"].notna().all()
print("label2id:", label2id)
print("Empty cleaned comments:", int((train_df["clean_text"] == "").sum()))

## 3. Tokenization

The validation folds are created from row indices. Tokenized datasets contain only model inputs and labels, preventing accidental leakage of the original test labels.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def tokenize_batch(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)


def make_dataset(frame: pd.DataFrame, include_labels: bool) -> Dataset:
    columns = ["clean_text"] + (["target"] if include_labels else [])
    dataset = Dataset.from_pandas(
        frame[columns].rename(columns={"target": "labels"}),
        preserve_index=False,
    )
    return dataset.map(
        tokenize_batch,
        batched=True,
        remove_columns=["clean_text"],
    )


test_tok = make_dataset(test_df, include_labels=False)
print(test_tok)

## 4. Regularized weighted Trainer

Each fold starts from the same public IndoBERTweet checkpoint. The model configuration increases dropout, while the loss adds class weighting and label smoothing. Early stopping selects the best epoch using Macro F1.

In [ ]:
def softmax_np(logits):
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.asarray(logits)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def build_model():
    config = AutoConfig.from_pretrained(MODEL_NAME)
    config.num_labels = len(label2id)
    config.id2label = id2label
    config.label2id = label2id
    config.classifier_dropout = CLASSIFIER_DROPOUT
    config.hidden_dropout_prob = HIDDEN_DROPOUT
    config.attention_probs_dropout_prob = ATTENTION_DROPOUT
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config,
    )


class RegularizedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device)
        loss_fn = nn.CrossEntropyLoss(
            weight=weights,
            label_smoothing=LABEL_SMOOTHING,
        )
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    labels = eval_pred.label_ids
    return {
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
        "accuracy": float((predictions == labels).mean()),
    }


def make_training_args(output_dir: str, fold: int):
    kwargs = dict(
        output_dir=output_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        max_grad_norm=MAX_GRAD_NORM,
        optim="adamw_torch",
        save_total_limit=1,
        report_to="none",
        seed=SEED + fold,
        data_seed=SEED + fold,
    )
    try:
        return TrainingArguments(eval_strategy="epoch", **kwargs)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kwargs)

## 5. Five-fold training and probability averaging

`oof_probs` gives one honest validation prediction for every labeled training row. `test_probs` averages the five fold models and is used for the competition submission and dashboard export.

Only the best-scoring fold model is kept as a single-model deployment artifact. The exported test predictions still come from the full five-fold ensemble.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
best_model_dir = EXPORT_DIR / "indobertweet_model"
fold_checkpoint_root = EXPORT_DIR / "fold_checkpoints"
fold_checkpoint_root.mkdir(parents=True, exist_ok=True)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_probs = np.zeros((len(train_df), len(label2id)), dtype=np.float32)
test_probs = np.zeros((len(test_df), len(label2id)), dtype=np.float32)
fold_rows = []
best_fold = None
best_fold_f1 = -np.inf

for fold, (train_indices, valid_indices) in enumerate(
    skf.split(train_df, train_df["target"]),
    start=1,
):
    print(f"\n========== FOLD {fold}/{N_SPLITS} ==========")
    fold_train = train_df.iloc[train_indices].reset_index(drop=True)
    fold_valid = train_df.iloc[valid_indices].reset_index(drop=True)

    fold_train_tok = make_dataset(fold_train, include_labels=True)
    fold_valid_tok = make_dataset(fold_valid, include_labels=True)

    fold_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(len(label2id)),
        y=fold_train["target"].to_numpy(),
    )
    fold_weights = torch.tensor(fold_weights, dtype=torch.float)
    fold_model = build_model()
    fold_output_dir = fold_checkpoint_root / f"fold_{fold}"

    fold_trainer = RegularizedTrainer(
        model=fold_model,
        args=make_training_args(str(fold_output_dir), fold),
        train_dataset=fold_train_tok,
        eval_dataset=fold_valid_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=fold_weights,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    fold_trainer.train()

    valid_output = fold_trainer.predict(fold_valid_tok)
    valid_probs = softmax_np(valid_output.predictions)
    oof_probs[valid_indices] = valid_probs
    valid_pred_ids = valid_probs.argmax(axis=1)
    fold_f1 = f1_score(
        fold_valid["target"].to_numpy(),
        valid_pred_ids,
        average="macro",
    )

    test_output = fold_trainer.predict(test_tok)
    test_probs += softmax_np(test_output.predictions) / N_SPLITS

    fold_rows.append({
        "fold": fold,
        "macro_f1": float(fold_f1),
        "best_epoch": int(round(fold_trainer.state.epoch or 0)),
        "training_steps": int(fold_trainer.state.global_step),
        "validation_rows": len(fold_valid),
    })
    print(f"Fold {fold} Macro F1: {fold_f1:.5f}")

    if fold_f1 > best_fold_f1:
        best_fold_f1 = float(fold_f1)
        best_fold = fold
        if best_model_dir.exists():
            shutil.rmtree(best_model_dir)
        fold_trainer.save_model(str(best_model_dir))
        tokenizer.save_pretrained(str(best_model_dir))

    if fold_output_dir.exists():
        shutil.rmtree(fold_output_dir)
    del fold_trainer, fold_model, fold_train_tok, fold_valid_tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

fold_metrics = pd.DataFrame(fold_rows)
fold_metrics.to_csv(EXPORT_DIR / "fold_metrics_v2.csv", index=False)
print("\nBest fold:", best_fold, "Macro F1:", best_fold_f1)
print("Mean fold Macro F1:", fold_metrics["macro_f1"].mean())
print("Std fold Macro F1:", fold_metrics["macro_f1"].std())

## 6. Out-of-fold evaluation and export

The out-of-fold score is the main v2 validation result because every training row is evaluated by a model that did not train on that row. The competition test set remains unlabeled.

In [ ]:
oof_pred_ids = oof_probs.argmax(axis=1)
oof_true_ids = train_df["target"].to_numpy()
oof_macro_f1 = f1_score(oof_true_ids, oof_pred_ids, average="macro")
oof_labels = [id2label[int(value)] for value in oof_pred_ids]
true_labels = [id2label[int(value)] for value in oof_true_ids]

print("OOF Macro F1:", round(float(oof_macro_f1), 6))
print(
    classification_report(
        oof_true_ids,
        oof_pred_ids,
        labels=list(range(len(id2label))),
        target_names=[id2label[i] for i in range(len(id2label))],
        digits=4,
        zero_division=0,
    )
)

oof_export = train_df[[ID_COL, TEXT_COL, LABEL_COL]].copy()
oof_export["true_label_id"] = oof_true_ids
oof_export["true_label"] = true_labels
oof_export["predicted_label_id"] = oof_pred_ids
oof_export["predicted_label"] = oof_labels
for class_id, class_name in id2label.items():
    oof_export[f"probability_{class_name}"] = oof_probs[:, class_id]
oof_export.to_csv(EXPORT_DIR / "predictions_oof_v2.csv", index=False, encoding="utf-8")

test_pred_ids = test_probs.argmax(axis=1)
test_pred_labels = [id2label[int(value)] for value in test_pred_ids]
test_export = test_df[[ID_COL, TEXT_COL]].copy()
test_export["predicted_label_id"] = test_pred_ids
test_export["predicted_label"] = test_pred_labels
for class_id, class_name in id2label.items():
    test_export[f"probability_{class_name}"] = test_probs[:, class_id]
test_export.to_csv(EXPORT_DIR / "predictions_test_v2.csv", index=False, encoding="utf-8")

report = classification_report(
    oof_true_ids,
    oof_pred_ids,
    labels=list(range(len(id2label))),
    target_names=[id2label[i] for i in range(len(id2label))],
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).transpose().to_csv(EXPORT_DIR / "classification_report_v2.csv")

cm = confusion_matrix(oof_true_ids, oof_pred_ids, labels=list(range(len(id2label))))
pd.DataFrame(
    cm,
    index=[id2label[i] for i in range(len(id2label))],
    columns=[id2label[i] for i in range(len(id2label))],
).to_csv(EXPORT_DIR / "confusion_matrix_v2.csv")

with open(EXPORT_DIR / "label_mapping_v2.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "id2label": {str(key): value for key, value in id2label.items()},
            "label2id": label2id,
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

metadata = {
    "model_version": "indobertweet_mbg_v2",
    "model_type": "5-fold regularized probability ensemble",
    "base_model": MODEL_NAME,
    "id_column": ID_COL,
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "label_mapping": {str(key): value for key, value in id2label.items()},
    "n_splits": N_SPLITS,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "num_epochs_max": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "max_grad_norm": MAX_GRAD_NORM,
    "classifier_dropout": CLASSIFIER_DROPOUT,
    "hidden_dropout": HIDDEN_DROPOUT,
    "attention_dropout": ATTENTION_DROPOUT,
    "best_fold": best_fold,
    "best_fold_macro_f1": float(best_fold_f1),
    "oof_macro_f1": float(oof_macro_f1),
    "train_rows": len(train_df),
    "competition_test_rows": len(test_df),
    "test_labels_available": False,
    "ensemble_prediction_method": "mean of five fold softmax probabilities",
}
with open(EXPORT_DIR / "model_metadata_v2.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

if "submission" in globals():
    submission = submission.copy()
    submission["label"] = test_pred_labels
else:
    submission = pd.DataFrame({ID_COL: test_df[ID_COL].values, "label": test_pred_labels})
submission.to_csv(EXPORT_DIR / "submission_v2.csv", index=False)

shutil.make_archive(
    str(EXPORT_DIR),
    "zip",
    root_dir=str(EXPORT_DIR.parent),
    base_dir=EXPORT_DIR.name,
)

print("\nExport complete:", EXPORT_DIR)
for path in sorted(EXPORT_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(EXPORT_DIR))

## Acceptance checklist

- The reported v2 validation score is the out-of-fold Macro F1, not the hidden Kaggle test score.
- `predictions_test_v2.csv` contains ensemble predictions but no fabricated test labels.
- `indobertweet_model/` is the best single fold for optional live inference; the CSV predictions use all five folds.
- Compare v2 against v1 on the same out-of-fold or held-out protocol before claiming improvement.
- Upload the exported CSV/JSON bundle to the Streamlit v2 dashboard. Do not describe it as a live opinion monitor without a current inference service.